In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(LSTMModel, self).__init__()
        self.hidden_size = hidden_size
        self.lstm = nn.LSTM(input_size, hidden_size)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, input):
        lstm_out, _ = self.lstm(input)
        output = self.fc(lstm_out[:, -1, :])  # taking only the last time step output
        return output

class ChannelPrediction: 
    def __init__(self, H_b, t, sequence_length, BS_NO, USER_NO): # default sequence length = 3
        self.H_b = H_b #  T * BS_NO * USER_NO
        self.t = t
        self.sequence_length = sequence_length
        self.BS_NO = BS_NO
        self.USER_NO = USER_NO

    def train_model(self):
        model = LSTMModel(input_size=self.BS_NO * self.USER_NO,
                          hidden_size=50,
                          output_size=self.BS_NO * self.USER_NO)
        
        loss_function = nn.MSELoss()
        optimizer = optim.Adam(model.parameters(), lr=0.001)


        for epoch in range(200):
            for t in range(self.sequence_length, self.t):
                data = torch.tensor(self.H_b[t-self.sequence_length:t], dtype=torch.float32).unsqueeze(0)
                labels = torch.tensor(self.H_b[t], dtype=torch.float32).unsqueeze(0)

                optimizer.zero_grad()
                output = model(data)
                loss = loss_function(output, labels)
                loss.backward()
                optimizer.step()

    
    def predict_next_channel(self):
        model = LSTMModel(input_size=self.BS_NO * self.USER_NO,
                          hidden_size=50,
                          output_size=self.BS_NO * self.USER_NO)
        # Load trained model weights
        #model.load_state_dict(torch.load('lstm_model_weights.pth'))
        # Select the sequence of data from H_b and reshape it to match the expected input shape
        data = torch.tensor(self.H_b[self.t-self.sequence_length+1:self.t+1], dtype=torch.float32)
        data = data.view(1, self.sequence_length, self.BS_NO * self.USER_NO)

        next_channel = model(data)

        #next_channel = model(torch.tensor(self.H_b[self.t-self.sequence_length+1:self.t+1], dtype=torch.float32).unsqueeze(0))
        return next_channel.detach().numpy()


T=10
BS_NO=2
USER_NO=5
sequence_length=3
H_b = np.zeros((T, BS_NO, USER_NO))

for t in range(sequence_length, T):
    H_b[t, :, :] = np.random.rand(BS_NO, USER_NO)
    channel_prediction = ChannelPrediction(H_b, t, sequence_length, BS_NO, USER_NO)
    channel_prediction.train_model()

    next_channel = channel_prediction.predict_next_channel()
    print(next_channel)

RuntimeError: input must have 3 dimensions, got 4